# LAB: Chaos Agent: Tools, Lore, Questionable Judgment

## Objective

Build a **creative complaint handler** for *NormalObjects* with a LangChain agent.
The agent picks its own tools, so you must **prove** the answer comes from tool output.

```
complaint → agent (LLM) → picks tools (any order) → tool outputs → final answer
```

## Steps

| # | Step | Code |
|---|------|---|
| 1 | Setup | PROVIDED |
| 2 | Create creative tools | PROVIDED + YOUR TURN |
| 3 | Create the agent | YOUR TURN |
| 4 | Test with sample complaints | YOUR TURN |
| 5 | Analyze agent behavior | YOUR TURN |
| | `lab_proof.md` | YOUR TURN |

**Submit:** `lab_proof.md` with code, one complaint, tool evidence, final output, one failure.

| Label | Meaning |
|---|---|
| **PROVIDED** | Run and read it |
| **YOUR TURN** | Replace the `# TODO` lines |
| **OPTIONAL** | Not required |

**Reference:** `langchain_integration.ipynb` (tools + `create_agent`)

**Prompt discipline:** use AI to review or debug, not to write your answer.

---
## Step 1: Setup [PROVIDED]

Add `OPENAI_API_KEY=...` to a `.env` file next to this notebook.

In [1]:
# Install required packages (uncomment to run)
%pip install -U langchain langchain-openai python-dotenv

  Using cached websockets-16.1.1-cp311-cp311-win_amd64.whl.metadata (7.0 kB)
Using cached websockets-16.1.1-cp311-cp311-win_amd64.whl (180 kB)

  Attempting uninstall: websockets

    Found existing installation: websockets 17.3

    Uninstalling websockets-17.3:

      Successfully uninstalled websockets-17.3

   ---------------------------------------- 0/2 [websockets]
   ---------------------------------------- 0/2 [websockets]
   ---------------------------------------- 0/2 [websockets]
   ---------------------------------------- 0/2 [websockets]
  Attempting uninstall: langchain
   ---------------------------------------- 0/2 [websockets]
    Found existing installation: langchain 1.4.6
   ---------------------------------------- 0/2 [websockets]
    Uninstalling langchain-1.4.6:
   ---------------------------------------- 0/2 [websockets]
      Successfully uninstalled langchain-1.4.6
   ---------------------------------------- 0/2 [websockets]
   -------------------- -----------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.66.1 requires pyarrow!=25.0.0,<27,>=10.0.1, which is not installed.
streamlit 1.66.1 requires watchdog<7,>=2.1.5; platform_system != "Darwin", which is not installed.


In [2]:
import os
import random 
from dotenv import load_dotenv

from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

load_dotenv()

# Make sure OPENAI_API_KEY is set in your .env file
print("OpenAI API Key loaded:", "OPENAI_API_KEY" in os.environ)

model = init_chat_model(model="gpt-4o-mini", model_provider="openai", temperature=0.7)

OpenAI API Key loaded: True


**Done when:** the key prints `True` and there are no import errors.

---
## Step 2: Create Creative Tools

Every tool needs:
1. the `@tool` decorator
2. a clear docstring (the agent reads it to choose the tool)

### 2a. `consult_demogorgon` [PROVIDED]

In [3]:
@tool
def consult_demogorgon(complaint: str) -> str:
    """Consult the Demogorgon about a complaint involving the Upside Down.
    Returns a chaotic, unreliable opinion."""
    responses = [
        f"The Demogorgon sniffs '{complaint}' and roars. It seems... hungry, not helpful.",
        f"The Demogorgon tilts its petal-face at '{complaint}'. Interpreted as: 'turn off the lights'.",
        f"The Demogorgon ignores '{complaint}' and walks back into the wall. Classic.",
    ]
    return random.choice(responses)

### 2b. `check_hawkins_records` [YOUR TURN]

- Dictionary with 4 keys: `portal`, `monsters`, `psychics`, `electricity`
- Return the record whose key appears in the query (case-insensitive)
- No match → `"No record found ..."`

In [4]:
# YOUR TURN: Hawkins records tool

@tool
def check_hawkins_records(query: str) -> str:
    """Search Hawkins records for evidence about portals,
    monsters, psychics, or electricity."""

    records = {
        "portal": "Portal activity is associated with weak barriers between worlds.",
        "monsters": "Demogorgons may hunt alone or respond to a hive mind.",
        "psychics": "Psychic abilities require energy, especially for heavy objects.",
        "electricity": "Nearby creatures can cause lights to flicker and electrical disturbances."
    }
    for key, record in records.items():
        if key in query.lower():
            return record

    return "No record found for this query."

In [5]:
print(check_hawkins_records.invoke({"query": "What causes PORTAL activity?"}))

Portal activity is associated with weak barriers between worlds.


### 2c. `cast_interdimensional_spell` [YOUR TURN]

- Args: `problem`, `creativity_level="medium"`
- Map `low/medium/high` → `1/2/3`
- 4 spell strings that include the problem
- `random.sample()` N spells, join with `"\n"`

In [6]:
# YOUR TURN: spell tool

@tool
def cast_interdimensional_spell(
    problem: str,
    creativity_level: str = "medium"
) -> str:
    """Suggest creative interdimensional spells for a problem.
    Creativity level can be low, medium, or high."""

    creativity_map = {
        "low": 1,
        "medium": 2,
        "high": 3
    }

    multiplier = creativity_map.get(creativity_level.lower(), 2)
    spells = [
        f"Cast a protective shield to contain the problem: {problem}",
        f"Open a temporary portal to investigate the problem: {problem}",
        f"Summon an interdimensional guardian to handle: {problem}",
        f"Use a magical energy field to stabilise: {problem}"
    ]
    selected_spells = random.sample(
        spells,
        min(multiplier, len(spells))
    )

    return "\n".join(selected_spells)

In [7]:
result = cast_interdimensional_spell.invoke({
    "problem": "The lights keep flickering",
    "creativity_level": "high"
})

print(result)

Use a magical energy field to stabilise: The lights keep flickering
Open a temporary portal to investigate the problem: The lights keep flickering
Summon an interdimensional guardian to handle: The lights keep flickering


### 2d. `gather_party_wisdom` [YOUR TURN]

- Dictionary with 4 keys, one per party member (Mike, Dustin, Lucas, Will)
- Return the match (case-insensitive)
- No match → generic "the party huddles..." answer

In [8]:
# YOUR TURN: party wisdom tool

@tool
def gather_party_wisdom(question: str) -> str:
    """Ask Mike, Dustin, Lucas, or Will for advice about a problem."""

    wisdom = {
        "mike": "Mike says: We need a plan. Let's work together.",
        "dustin": "Dustin says: Let's investigate and look for scientific clues.",
        "lucas": "Lucas says: Be careful. We should assess the risks first.",
        "will": "Will says: I can sense when something strange is happening."
    }
    for member, advice in wisdom.items():
        if member in question.lower():
            return advice

    return "The party huddles together to discuss the problem, but no specific advice is available."

In [9]:
# YOUR TURN: test each tool on its own with .invoke({...})

print(consult_demogorgon.invoke({"complaint": "My lights keep flickering"}))

print(check_hawkins_records.invoke({"query": "tell me about the portal"}))

print(cast_interdimensional_spell.invoke({
    "problem": "Demodogs in my garden",
    "creativity_level": "high"
}))

print(gather_party_wisdom.invoke({"question": "what is the plan?"}))

The Demogorgon tilts its petal-face at 'My lights keep flickering'. Interpreted as: 'turn off the lights'.
Portal activity is associated with weak barriers between worlds.
Use a magical energy field to stabilise: Demodogs in my garden
Open a temporary portal to investigate the problem: Demodogs in my garden
Summon an interdimensional guardian to handle: Demodogs in my garden
The party huddles together to discuss the problem, but no specific advice is available.


**Done when:** all 4 tools return text, and a query with no matching key returns your fallback.

---
## Step 3: Create the Agent [YOUR TURN]

System prompt must:
- encourage creative tool use
- force at least one tool call
- say: answer **only** from tool results
- ask the agent to list the tools it used

**Hint:** `create_agent(model=..., tools=..., system_prompt=...)` in `langchain_integration.ipynb`.

In [10]:
# YOUR TURN: tool list + system prompt + agent

tools = [
    consult_demogorgon,
    check_hawkins_records,
    cast_interdimensional_spell,
    gather_party_wisdom
]
system_prompt = """
You are a creative complaint handler in the NormalObjects universe.

Use the available tools creatively to investigate and respond to complaints.

You MUST call at least one tool before giving your final answer.

Base your final answer only on information returned by the tools.
Do not invent facts or claim unsupported information as evidence.

You may combine results from multiple tools when appropriate.
Clearly distinguish fictional creative suggestions from factual records.

At the end of every final answer, list the names of the tools you used.
"""

agent = create_agent(
    model=model,
    tools=tools,
    system_prompt=system_prompt
)

print([t.name for t in tools])

['consult_demogorgon', 'check_hawkins_records', 'cast_interdimensional_spell', 'gather_party_wisdom']


**Done when:** the agent is created and prints the 4 tool names.

---
## Step 4: Test with Sample Complaints [YOUR TURN]

Use `config={"recursion_limit": 10}` to stop loops.

In [11]:
complaints = [
    "My Christmas lights keep flickering and I hear noises from the wall. What is going on?",
    "There is a weird portal in my basement. What's the plan?",
    "A Demodog ate my homework. I need a creative solution, creativity level high!",
    "My neighbor's cat is acting weird. Is it from the Upside Down?",  # no record exists
]

# YOUR TURN: run every complaint, keep the results for Step 5

results = []
for complaint in complaints:
    print("\nComplaint:", complaint)

    result = agent.invoke(
        {"messages": [HumanMessage(content=complaint)]},
        config={"recursion_limit": 10}
    )

    results.append(result)


Complaint: My Christmas lights keep flickering and I hear noises from the wall. What is going on?

Complaint: There is a weird portal in my basement. What's the plan?

Complaint: A Demodog ate my homework. I need a creative solution, creativity level high!

Complaint: My neighbor's cat is acting weird. Is it from the Upside Down?


**Check:** for the cat complaint, does the agent admit "no record", or invent one?

In [12]:
for i, result in enumerate(results, start=1):
    print(f"\nCOMPLAINT {i}:")
    print(complaints[i - 1])

    print("\nFINAL ANSWER:")
    print(result["messages"][-1].content)

    print("-" * 60)


COMPLAINT 1:
My Christmas lights keep flickering and I hear noises from the wall. What is going on?

FINAL ANSWER:
It appears that I couldn't find any records in Hawkins related to flickering Christmas lights or noises coming from the wall. The Demogorgon, however, had a rather ominous reaction, suggesting that it might be hungry but providing no useful information.

Given these findings, here are a couple of creative suggestions to address your situation:

1. **Check the Wiring**: Ensure that the electrical connections for your Christmas lights are secure. Loose wiring can cause flickering and might be the source of the strange noises.

2. **Consult an Expert**: Since the Demogorgon didn't provide any constructive advice, perhaps it’s best to call an electrician to investigate any potential electrical issues or other structural concerns in your wall.

If you suspect any supernatural interference, it might be wise to keep an eye out for any strange occurrences, just in case.

Tools us

In [13]:
print("COMPLAINT 4:")
print(complaints[3])

print("\nFINAL ANSWER:")
print(results[3]["messages"][-1].content)

COMPLAINT 4:
My neighbor's cat is acting weird. Is it from the Upside Down?

FINAL ANSWER:
It seems there are no specific records regarding unusual cat behavior related to the Upside Down in Hawkins. However, the Demogorgon suggests a rather cryptic response: "turn off the lights." This could imply that the cat's strange behavior might be influenced by changes in its environment, such as lighting. 

If the cat is acting particularly erratic, consider checking if there are any disturbances in the area, or perhaps even giving it a more relaxed environment by dimming the lights. 

Feel free to observe the cat further and see if the behavior changes!

Tools used: check_hawkins_records, consult_demogorgon.


---
## Step 5: Analyze Agent Behavior [YOUR TURN]

### 5a. Track tool usage

For each result: print the tool order (`a → b → c`). Then count total uses per tool.

**Hint:** AI messages have `.tool_calls`; use `collections.Counter`.

In [15]:
# YOUR TURN: tool order + counts

from collections import Counter

tool_counts = Counter()
for result in results:
    order = []
    for msg in result["messages"]:
        if msg.type == "ai":
            for call in msg.tool_calls:
                order.append(call["name"])
    tool_counts.update(order)
    print(" → ".join(order))
print("\nTotal tool usage:", tool_counts)


check_hawkins_records → consult_demogorgon
check_hawkins_records → gather_party_wisdom → consult_demogorgon
cast_interdimensional_spell → consult_demogorgon
check_hawkins_records → consult_demogorgon

Total tool usage: Counter({'consult_demogorgon': 4, 'check_hawkins_records': 3, 'gather_party_wisdom': 1, 'cast_interdimensional_spell': 1})


### 5b. Trace one answer (grounding)

For one result, print every tool call, its args, and what it returned.

**Hint:** messages with `msg.type == "tool"` hold the tool output.

In [20]:
# YOUR TURN: evidence trace for one result

# Tool outputs
for msg in results[0]["messages"]:
    if msg.type == "tool":
        print("Tool name:", msg.name)
        print("Tool output:", msg.content)
        print("-" * 60)

# Tool calls and arguments
print("\nTOOL CALLS:")

for msg in results[0]["messages"]:
    if msg.type == "ai" and msg.tool_calls:
        for call in msg.tool_calls:
            print("Tool:", call["name"])
            print("Arguments:", call["args"])
            print("-" * 60)

# Final answer
print("\nFINAL ANSWER:")
print(results[0]["messages"][-1].content)

Tool name: check_hawkins_records
Tool output: No record found for this query.
------------------------------------------------------------
Tool name: consult_demogorgon
Tool output: The Demogorgon sniffs 'Christmas lights flickering and noises from the wall' and roars. It seems... hungry, not helpful.
------------------------------------------------------------

TOOL CALLS:
Tool: check_hawkins_records
Arguments: {'query': 'flickering lights, noises from the wall'}
------------------------------------------------------------
Tool: consult_demogorgon
Arguments: {'complaint': 'Christmas lights flickering and noises from the wall'}
------------------------------------------------------------

FINAL ANSWER:
It appears that I couldn't find any records in Hawkins related to flickering Christmas lights or noises coming from the wall. The Demogorgon, however, had a rather ominous reaction, suggesting that it might be hungry but providing no useful information.

Given these findings, here are a 

**Check:** mark each claim in the final answer as either from a tool or invented.

### 5c. Compare with a structured approach

Write `structured_handler(complaint)`: always calls records → party → spell (`low`), no LLM.

In [21]:
# YOUR TURN: fixed-order handler

def structured_handler(complaint: str) -> str:
    records = check_hawkins_records.invoke({"query": complaint})
    party = gather_party_wisdom.invoke({"question": complaint})
    spell = cast_interdimensional_spell.invoke({
        "problem": complaint,
        "creativity_level": "low"
    })
    return (
        f"Hawkins Records: {records}\n"
        f"Party Wisdom: {party}\n"
        f"Spell: {spell}"
    )
print("STRUCTURED HANDLER RESULT:")
print(structured_handler(complaints[1]))

STRUCTURED HANDLER RESULT:
Hawkins Records: Portal activity is associated with weak barriers between worlds.
Party Wisdom: The party huddles together to discuss the problem, but no specific advice is available.
Spell: Summon an interdimensional guardian to handle: There is a weird portal in my basement. What's the plan?



| Question | Agent | Structured |
|---|---|---|
| Same tools every run? | No, tools depend on the complaint | Yes, always the same three tools |
| Tool order predictable? | No, the agent decides the order | Yes, records → party → spell |
| Can invent facts? | Yes, the LLM may add unsupported claims | No LLM-generated claims, but predefined spell suggestions are fictional |
| Easy to trace? | Requires inspecting tool calls and outputs | Yes, fixed sequence and direct outputs |
| Best for which client case? | Creative, flexible complaint handling | Controlled, repeatable processes requiring predictable results |


---
## Extensions [OPTIONAL]

- More tools (`consult_eleven`, `check_government_files`)
- LLM call inside a tool
- Memory with `InMemorySaver` + `thread_id`
- Streamlit / Gradio UI
- Measure response time and tool calls per complaint

In [ ]:
# YOUR TURN (optional)

# TODO

---
## Write `lab_proof.md` [YOUR TURN]

```markdown
# Lab Proof: NormalObjects Creative Complaint Handler

## 1. What I built
## 2. Query
## 3. Tool evidence (calls + args + outputs)
## 4. Final output
## 5. Trace (which claim comes from which tool output?)
## 6. Failure or limitation (where can it give a plausible but unsupported answer?)
## 7. Agent vs. structured handler
```

**Before you submit:**
- [ ] No API keys or `.env`
- [ ] Only lab files
- [ ] `lab_proof.md` is the entry point

---
### Troubleshooting

| Problem | Fix |
|---|---|
| Agent doesn't use tools | Clearer docstrings; prompt must require tool use |
| Tools in "wrong" order | Expected for agents. Need fixed order? Use LangGraph |
| Agent loops | Lower `recursion_limit`, improve docstrings |
| Generic answers | More specific tool outputs, lower temperature |
| `ImportError` on `create_agent` | `pip install -U langchain` (needs v1+) |